# 02 - Procesamiento y Feature Engineering
## Sistema Inteligente de Predicción de Riesgo para Deportistas Amateurs

**Objetivo:** Preparar el dataset final para entrenar los modelos ML. Incluye encoding de categóricas, generación de features sintéticas de sueño y nutrición, corrección del desbalance con SMOTE, normalización y división train/val/test.

## 1. Importaciones y carga del dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='husl')

df = pd.read_csv('../data/raw/collegiate_athlete_injury_dataset.csv')
print(f'Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

## 2. Encoding de variables categóricas

Los modelos ML no entienden texto. Convertimos `Gender` y `Position` a números mediante **One-Hot Encoding**: cada categoría se convierte en una columna binaria (0/1).

In [ ]:
# Eliminar columna ID (no aporta información predictiva)
df = df.drop(columns=['Athlete_ID'])

# One-Hot Encoding de Gender y Position
df = pd.get_dummies(df, columns=['Gender', 'Position'], drop_first=False, dtype=int)

print(f'Columnas tras encoding: {df.shape[1]}')
print('\nNuevas columnas generadas:')
for col in df.columns:
    if col.startswith('Gender_') or col.startswith('Position_'):
        print(f'  - {col}: {df[col].sum()} atletas')
df.head(3)

## 3. Generación de features sintéticas de sueño y nutrición

El dataset original no incluye datos de sueño ni nutrición. Los generamos sintéticamente con distribuciones realistas basadas en literatura científica (OMS, ACSM) y correlaciones lógicas con las variables existentes.

| Feature sintética | Correlación usada | Justificación |
|---|---|---|
| `sleep_hours` | Inversa con `Fatigue_Score` | Más fatiga → menos sueño |
| `sleep_quality` | Positiva con `Recovery_Days_Per_Week` | Más recuperación → mejor sueño |
| `sleep_deficit` | Derivada de `sleep_hours` | Déficit si duerme < 7h (recomendación OMS) |
| `meals_per_day` | Positiva con `Performance_Score` | Mejor rendimiento → mejor nutrición |
| `hydration_liters` | Positiva con `Training_Hours_Per_Week` | Más entrenamiento → más hidratación necesaria |

In [ ]:
n = len(df)

# sleep_hours: base 7.5h, reducida por fatiga (rango 4-10h)
fatiga_norm = (df['Fatigue_Score'] - df['Fatigue_Score'].min()) / (df['Fatigue_Score'].max() - df['Fatigue_Score'].min())
sleep_base = 8.5 - (fatiga_norm * 3.0)
df['sleep_hours'] = np.clip(sleep_base + np.random.normal(0, 0.5, n), 4.0, 10.0).round(1)

# sleep_quality: base 6/10, mejorada por días de recuperación (rango 1-10)
recov_norm = (df['Recovery_Days_Per_Week'] - df['Recovery_Days_Per_Week'].min()) / (df['Recovery_Days_Per_Week'].max() - df['Recovery_Days_Per_Week'].min())
quality_base = 4.0 + (recov_norm * 5.0)
df['sleep_quality'] = np.clip(quality_base + np.random.normal(0, 0.8, n), 1.0, 10.0).round(1)

# sleep_deficit: horas por debajo de 7 acumuladas (0 si duerme >= 7h)
df['sleep_deficit'] = np.maximum(0, 7.0 - df['sleep_hours']).round(1)

# meals_per_day: entre 2 y 6, correlacionada con rendimiento
perf_norm = (df['Performance_Score'] - df['Performance_Score'].min()) / (df['Performance_Score'].max() - df['Performance_Score'].min())
meals_base = 2.5 + (perf_norm * 3.0)
df['meals_per_day'] = np.clip(np.round(meals_base + np.random.normal(0, 0.4, n)), 2, 6).astype(int)

# hydration_liters: entre 1 y 4L, correlacionada con horas de entrenamiento
hours_norm = (df['Training_Hours_Per_Week'] - df['Training_Hours_Per_Week'].min()) / (df['Training_Hours_Per_Week'].max() - df['Training_Hours_Per_Week'].min())
hydration_base = 1.5 + (hours_norm * 2.5)
df['hydration_liters'] = np.clip(hydration_base + np.random.normal(0, 0.3, n), 1.0, 4.0).round(1)

print('Features sintéticas generadas:')
print(df[['sleep_hours','sleep_quality','sleep_deficit','meals_per_day','hydration_liters']].describe().round(2))

In [ ]:
# Verificación visual de las features sintéticas
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
features_sinteticas = ['sleep_hours', 'sleep_quality', 'sleep_deficit', 'meals_per_day', 'hydration_liters']
colores = ['#8e44ad', '#2980b9', '#c0392b', '#27ae60', '#f39c12']

for i, (col, color) in enumerate(zip(features_sinteticas, colores)):
    axes[i].hist(df[col], bins=20, color=color, edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Media: {df[col].mean():.1f}')
    axes[i].legend(fontsize=8)

plt.suptitle('Distribución de features sintéticas generadas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/feat_01_sinteticas.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nDataset con features sintéticas: {df.shape[1]} columnas totales')

## 4. Separación de features y target

In [ ]:
X = df.drop(columns=['Injury_Indicator'])
y = df['Injury_Indicator']

print(f'Features (X): {X.shape[1]} columnas')
print(f'Target (y): {y.shape[0]} filas')
print(f'\nDistribución del target antes de SMOTE:')
print(f'  Sin lesión (0): {(y==0).sum()} ({(y==0).mean()*100:.1f}%)')
print(f'  Con lesión (1): {(y==1).sum()} ({(y==1).mean()*100:.1f}%)')
print(f'\nColumnas de entrada:')
for col in X.columns:
    print(f'  - {col}')

## 5. Corrección del desbalance con SMOTE

**SMOTE** (Synthetic Minority Over-sampling Technique) genera ejemplos sintéticos de la clase minoritaria (atletas con lesión) interpolando entre casos reales existentes. Así equilibramos las clases sin perder información.

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print('Distribución del target DESPUÉS de SMOTE:')
print(f'  Sin lesión (0): {(y_resampled==0).sum()} ({(y_resampled==0).mean()*100:.1f}%)')
print(f'  Con lesión (1): {(y_resampled==1).sum()} ({(y_resampled==1).mean()*100:.1f}%)')
print(f'\nTotal de muestras: {len(y_resampled)} (antes: {len(y)})')

# Visualización antes/después
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colores = ['#2ecc71', '#e74c3c']

axes[0].bar(['Sin lesión','Con lesión'], [( y==0).sum(),(y==1).sum()], color=colores, edgecolor='white')
axes[0].set_title('Antes de SMOTE', fontweight='bold')
axes[0].set_ylabel('Número de muestras')
for i, v in enumerate([(y==0).sum(),(y==1).sum()]):
    axes[0].text(i, v+1, str(v), ha='center', fontweight='bold')

axes[1].bar(['Sin lesión','Con lesión'], [(y_resampled==0).sum(),(y_resampled==1).sum()], color=colores, edgecolor='white')
axes[1].set_title('Después de SMOTE', fontweight='bold')
axes[1].set_ylabel('Número de muestras')
for i, v in enumerate([(y_resampled==0).sum(),(y_resampled==1).sum()]):
    axes[1].text(i, v+1, str(v), ha='center', fontweight='bold')

plt.suptitle('Efecto de SMOTE sobre el balance de clases', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/feat_02_smote.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Normalización con StandardScaler

Escalamos todas las features para que tengan media 0 y desviación estándar 1. Esto evita que variables con valores grandes (como `Training_Hours_Per_Week`) dominen sobre variables pequeñas (como `sleep_deficit`).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_resampled)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print('Estadísticas tras normalización (deben ser ~0 media y ~1 std):')
print(X_scaled.describe().round(2).loc[['mean','std']])

## 7. División train / validation / test (60 / 20 / 20)

In [ ]:
# Primera división: 80% train+val / 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_scaled, y_resampled, test_size=0.20, random_state=42, stratify=y_resampled
)

# Segunda división: 75% train / 25% val (= 60% y 20% del total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print('División de datos:')
print(f'  Train:      {len(X_train):4d} muestras ({len(X_train)/len(X_scaled)*100:.0f}%)')
print(f'  Validation: {len(X_val):4d} muestras ({len(X_val)/len(X_scaled)*100:.0f}%)')
print(f'  Test:       {len(X_test):4d} muestras ({len(X_test)/len(X_scaled)*100:.0f}%)')
print(f'  Total:      {len(X_scaled):4d} muestras')
print(f'\nBalance en cada split:')
for nombre, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f'  {nombre}: {(y_split==0).sum()} sin lesión / {(y_split==1).sum()} con lesión')

## 8. Guardar datasets procesados

In [ ]:
import joblib

# Guardar splits como CSV
X_train.assign(Injury_Indicator=y_train.values).to_csv('../data/processed/train.csv', index=False)
X_val.assign(Injury_Indicator=y_val.values).to_csv('../data/processed/validation.csv', index=False)
X_test.assign(Injury_Indicator=y_test.values).to_csv('../data/processed/test.csv', index=False)

# Guardar el scaler para usarlo en la app Streamlit
joblib.dump(scaler, '../models/scaler.pkl')

# Guardar lista de columnas para la app
joblib.dump(list(X.columns), '../models/feature_columns.pkl')

print('Archivos guardados:')
print('  - data/processed/train.csv')
print('  - data/processed/validation.csv')
print('  - data/processed/test.csv')
print('  - models/scaler.pkl')
print('  - models/feature_columns.pkl')
print(f'\nColumnas del modelo ({len(X.columns)} features):')
for col in X.columns:
    print(f'  - {col}')

## 9. Resumen del procesamiento

In [ ]:
print('=' * 60)
print('RESUMEN DEL FEATURE ENGINEERING')
print('=' * 60)
print(f"""
TRANSFORMACIONES APLICADAS:
  1. Eliminado Athlete_ID (no predictivo)
  2. One-Hot Encoding: Gender y Position
  3. Features sintéticas añadidas:
     - sleep_hours      (media: {df['sleep_hours'].mean():.1f}h)
     - sleep_quality    (media: {df['sleep_quality'].mean():.1f}/10)
     - sleep_deficit    (media: {df['sleep_deficit'].mean():.1f}h de déficit)
     - meals_per_day    (media: {df['meals_per_day'].mean():.1f} comidas)
     - hydration_liters (media: {df['hydration_liters'].mean():.1f}L)
  4. SMOTE: {len(y)} → {len(y_resampled)} muestras (clases equilibradas)
  5. StandardScaler aplicado
  6. División 60/20/20

DATASET FINAL:
  - Features totales: {X.shape[1]}
  - Train:      {len(X_train)} muestras
  - Validation: {len(X_val)} muestras
  - Test:       {len(X_test)} muestras

SIGUIENTE PASO:
  Notebook 03_Modelos.ipynb: entrenar y comparar
  Regresión Logística, Random Forest, XGBoost y Ensemble.
""")